# Final Report — ASL Interpretation (WLASL)

*Team 13: Franky Liu, Jaron Zhu, Matthew Mitchell*

## 1. Data Pipeline

### Purpose
Convert the WLASL JSON into a clean tabular format for modeling and analysis, then ensure the corresponding videos exist locally.

### Pipeline Design
1. Load `WLASL_v0.3.json`.
2. Explode the `instances` list so each row is a *single video instance*.
3. Normalize instance fields (e.g., `video_id`, `frame_start`, `frame_end`, `bbox`).
4. Drop unused columns (e.g., URL if present).
5. Remove missing video IDs using `missing.txt`.
6. (Optional) join video metadata (fps, frame count, duration, resolution).

### Outputs
- `clean_df`: per-video instance table for training/EDA.
- `class_counts`: distribution of instances per gloss.
- `video_meta_df` (optional): per-video metadata used for EDA and sanity checks.


In [ ]:
import os
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import math, random

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import WeightedRandomSampler

import torchvision
import torchvision.transforms as tvt

try:
    import cv2
except Exception as e:
    cv2 = None
    print("cv2 not available:", e)

In [ ]:
# Old dataset we were originally going to use,
OLD_JSON_PATH = Path("dataset/WLASL_v0.3.json")
OLD_MISSING_PATH = Path("dataset/missing.txt")

OLD_VIDEO_DIR_CANDIDATES = [
    Path("wlasl/wlasl-complete/videos"),
    Path("dataset/videos"),
    Path("videos"),
]

def find_video_dir(candidates):
    for p in candidates:
        if p.exists():
            return p
    return None

OLD_VIDEO_DIR = find_video_dir(OLD_VIDEO_DIR_CANDIDATES)
print("JSON exists:", OLD_JSON_PATH.exists())
print("Missing list exists:", OLD_MISSING_PATH.exists())
print("Video dir found:", OLD_VIDEO_DIR)

In [ ]:
# Load OLD JSON
with open(OLD_JSON_PATH, "r") as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
exploded = df.explode("instances").reset_index(drop=True)
inst = pd.json_normalize(exploded["instances"])
clean_df = pd.concat([exploded[["gloss"]].reset_index(drop=True), inst], axis=1)

# Clean up columns we won't use right now
if "url" in clean_df.columns:
    clean_df = clean_df.drop(columns=["url"])

clean_df.head()

In [ ]:
if OLD_MISSING_PATH.exists():
    with open(OLD_MISSING_PATH, "r") as f:
        missing_ids = set([line.strip() for line in f.readlines() if line.strip()])
    before = len(clean_df)
    if "video_id" in clean_df.columns:
        clean_df = clean_df[~clean_df["video_id"].astype(str).isin(missing_ids)].copy()
    after = len(clean_df)
    print(f"Removed {before-after:,} rows due to missing videos ({len(missing_ids):,} missing IDs listed).")
else:
    print("No missing.txt found; skipping missing-video filtering.")

clean_df.shape

As you can see, in the old dataset we were orignally going to use, there are 9,103 missing videos and below is the new dataset we will use that has all the videos. 

In [ ]:
JSON_PATH = Path("clean videos/wlasl-complete/WLASL_v0.3.json")
MISSING_PATH = Path("clean videos/wlasl-complete/missing.txt")

VIDEO_DIR_CANDIDATES = [
    Path("clean videos/wlasl-complete/videos"),
]

def find_video_dir(candidates):
    for p in candidates:
        if p.exists():
            return p
    return None

VIDEO_DIR = find_video_dir(VIDEO_DIR_CANDIDATES)
print("JSON exists:", JSON_PATH.exists())
print("Missing list exists:", MISSING_PATH.exists())
print("Video dir found:", VIDEO_DIR)

In [ ]:
# Load JSON
with open(JSON_PATH, "r") as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
exploded = df.explode("instances").reset_index(drop=True)
inst = pd.json_normalize(exploded["instances"])
clean_df = pd.concat([exploded[["gloss"]].reset_index(drop=True), inst], axis=1)

# Clean up columns we won't use right now
if "url" in clean_df.columns:
    clean_df = clean_df.drop(columns=["url"])

clean_df.head()

In [ ]:
if MISSING_PATH.exists():
    with open(MISSING_PATH, "r") as f:
        missing_ids = set([line.strip() for line in f.readlines() if line.strip()])
    before = len(clean_df)
    if "video_id" in clean_df.columns:
        clean_df = clean_df[~clean_df["video_id"].astype(str).isin(missing_ids)].copy()
    after = len(clean_df)
    print(f"Removed {before-after:,} rows due to missing videos ({len(missing_ids):,} missing IDs listed).")
else:
    print("No missing.txt found; skipping missing-video filtering.")

clean_df.shape

There are 0 missing videos in the new dataset. 

## 2. Graph and Visual Exploration

### a. Top 20 most frequent gloss (words) in the dataset

In [ ]:
class_counts = clean_df["gloss"].value_counts()

top_n = 20
top = class_counts.head(top_n).iloc[::-1]

plt.figure(figsize=(10, 7))
plt.barh(top.index, top.values)
plt.title(f"Top 20 most frequent words")
plt.xlabel("Number of instances")
plt.ylabel("Gloss")
plt.tight_layout()
plt.show()

### b. Distribution of unique words in the dataset

In [ ]:
unique_glossary = clean_df['gloss'].unique()
print(f"There are {len(unique_glossary)} unique words in the glossary.")
print(f"A sample of the words... {unique_glossary}")

In [ ]:
total_count = clean_df['gloss'].value_counts()

plt.figure(figsize=(12,6))
plt.hist(total_count, bins=10, edgecolor='black')
plt.title("Distribution of word appearence")
plt.xlabel("Number of videos per word")
plt.ylabel("Number of glosses")
plt.show()

### c. Histogram of clip duration post processing json 

In [ ]:
def get_video_metadata(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    fps = cap.get(cv2.CAP_PROP_FPS) or np.nan
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT) or np.nan
    width = cap.get(cv2.CAP_PROP_FRAME_WIDTH) or np.nan
    height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or np.nan
    cap.release()
    duration = (frame_count / fps) if (fps and fps > 0 and frame_count and frame_count > 0) else np.nan
    return {
        "fps": fps,
        "frame_count": frame_count,
        "width": width,
        "height": height,
        "duration_s": duration,
    }

def resolve_video_path(video_id):
    if VIDEO_DIR is None:
        return None
    p = VIDEO_DIR / f"{video_id}.mp4"
    if p.exists():
        return p
    return None

In [ ]:
SAMPLE_N = 800
ids = clean_df["video_id"].astype(str).unique().tolist()
random.shuffle(ids)
ids = ids[:min(SAMPLE_N, len(ids))]

rows = []
missing_local = 0
for vid in ids:
    vp = resolve_video_path(vid)
    if vp is None:
        missing_local += 1
        continue
    meta = get_video_metadata(vp)
    if meta is None:
        continue
    meta["video_id"] = vid
    rows.append(meta)

video_meta_df = pd.DataFrame(rows)
print("Valid:", len(video_meta_df))
print("Missing:", missing_local)
video_meta_df.describe(include="all")

In [ ]:
dur = video_meta_df["duration_s"].dropna()
plt.figure(figsize=(10, 5))
plt.hist(dur, bins=40, edgecolor="black")
plt.title("Histogram of clip duration in seconds, from a sample of 800")
plt.xlabel("Duration in seconds")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

Also checking if frame count is consistent with the duration

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(video_meta_df["frame_count"],video_meta_df["duration_s"],alpha=0.6)
plt.xlabel("Frame count")
plt.ylabel("Duration in seconds")
plt.title("Video duration vs frame count from a sample of 800")
plt.tight_layout()
plt.show()

### d. Average video duration in words

In [ ]:
merged = clean_df.merge(video_meta_df[['video_id', 'duration_s']], on='video_id', how='left')
avg_duration = merged.groupby("gloss")["duration_s"].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
plt.plot(avg_duration.values)
plt.xlabel("Words sorted by its average duration")
plt.ylabel("Average duration in seconds")
plt.title("Average video duration per word, sample of 800")
plt.grid(True, alpha=0.5)
plt.tight_layout()
plt.show()

### d. Bias check on number of signers across the dataset

In [ ]:
signer_cols = [c for c in clean_df.columns if "signer" in c.lower()]

signer_col = signer_cols[0]
unique_signers_per_gloss = (clean_df.groupby("gloss")[signer_col].nunique().sort_values(ascending=False).to_frame("unique_signers"))
print(unique_signers_per_gloss.head(20))

In [ ]:
signer_counts = clean_df.groupby("gloss")[signer_col].nunique()

plt.figure(figsize=(10, 6))
plt.hist(signer_counts, bins=10, edgecolor="black")
plt.title("Signer diversity per word")
plt.xlabel("Number of unique signers")
plt.ylabel("Number of words")
plt.tight_layout()
plt.show()

## References

- Li, D., Rodriguez Opazo, C., Yu, X., & Li, H. (2020). *Word-level Deep Sign Language Recognition from Video: A New Large-scale Dataset and Methods Comparison*. WACV 2020.


## Table-based EDA (DataFrame Transformations)

These cells produce non-graphical EDA artifacts (tables) that highlight dataset structure, imbalance, and metadata availability.

### a. Column schema and spatial features (helps decide what features are feasible)

In [ ]:
schema_summary = pd.DataFrame({
    "dtype": clean_df.dtypes.astype(str),
    "non_null": clean_df.notna().sum(),
    "null": clean_df.isna().sum(),
    "null_%": (clean_df.isna().mean() * 100).round(2)
}).sort_values("null_%", ascending=False)

schema_summary

### b. Class imbalance in table form

In [ ]:
gloss_counts = clean_df["gloss"].value_counts()

top20_glosses = gloss_counts.head(20).to_frame("instances")
bottom20_glosses = gloss_counts.tail(20).to_frame("instances")

top20_glosses["type"] = "Top 20"
bottom20_glosses["type"] = "Bottom 20"
combined_df = pd.concat([top20_glosses, bottom20_glosses])

combined_df

### c. Bounding box coverage and shape sanitary check

In [ ]:
bbox_like_cols = [c for c in clean_df.columns if "bbox" in c.lower()]
print("Valid column exists?", bbox_like_cols)

def bbox_len(x):
    try:
        return len(x)
    except Exception:
        return np.nan

if "bbox" in clean_df.columns:
    pct_bbox_present = clean_df["bbox"].notna().mean() * 100
    print(f"% rows with bbox present: {pct_bbox_present:.2f}%")

    clean_df["bbox_len"] = clean_df["bbox"].apply(bbox_len)
    clean_df["bbox_len"].value_counts(dropna=False).to_frame("rows")

### e. Duplicate and leakage check

In [ ]:
dup_gloss_video = clean_df.duplicated(subset=["gloss", "video_id"]).sum()
print("Duplicate rows on (gloss, video_id):", int(dup_gloss_video))

vid_counts = clean_df["video_id"].astype(str).value_counts()
print("Unique video_ids:", int(vid_counts.shape[0]))
print("Video_ids reused:", int((vid_counts > 1).sum()))
print(vid_counts.head(10))

### f. One final, comprehensive look at the top 20 words and their metadata

In [ ]:
table_summary = clean_df.merge(video_meta_df[['video_id', 'duration_s']], on='video_id', how='left')

gloss_summary = table_summary.dropna(subset=["duration_s"]).groupby("gloss").agg(
    instances=("video_id", "count"),
    unique_signers=(signer_col, "nunique"),
    avg_duration=("duration_s", "mean"),
    min_duration=("duration_s", "min"),
    max_duration=("duration_s", "max")
).sort_values("instances", ascending=False)

gloss_summary.head(20)


---

# Bounding Box Baseline 



## 1) Video utilities: resolve path + uniform frame sampling

We sample **T** frames uniformly across the clip to preserve motion across the sign.

In [ ]:
# Define a constant to easily change the number of displayed frames
NUM_FRAMES_CONSTANT = 10

# Target model/input size (the new dataset uses bbox coords in this 256x256 space)
TARGET_SIZE = 256  # produces frames of shape (256, 256, 3)
FIXED_BBOX = (0, 0, TARGET_SIZE, TARGET_SIZE)

def resolve_video_path(video_id: str):
    if VIDEO_DIR is None:
        return None
    p = VIDEO_DIR / f"{video_id}.mp4"
    return p if p.exists() else None

def sample_frame_indices(num_frames: int, T: int):
    if num_frames <= 0:
        return []
    if num_frames < T:
        idx = np.linspace(0, num_frames - 1, num_frames).astype(int).tolist()
        idx += [num_frames - 1] * (T - num_frames)
        return idx
    return np.linspace(0, num_frames - 1, T).astype(int).tolist()

def read_frames_uniform(video_path, T=16, target_size=256):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return [], None, None

    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    frames = []
    orig_W = orig_H = None

    # indices we want (uniform)
    if n > 0:
        wanted = set(np.linspace(0, n-1, T).round().astype(int).tolist())
    else:
        wanted = None  # unknown length: just take first T

    i = 0
    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            break
        if orig_W is None:
            orig_H, orig_W = frame.shape[:2]

        take = (wanted is None and len(frames) < T) or (wanted is not None and i in wanted)
        if take:
            if frame.shape[0] != target_size or frame.shape[1] != target_size:
                frame = cv2.resize(frame, (target_size,target_size), interpolation=cv2.INTER_AREA)

            frames.append(frame)
            if wanted is None and len(frames) >= T:
                break
        i += 1

    cap.release()

    if len(frames) == 0:
        return [], orig_W, orig_H
    if len(frames) < T:
        frames.extend([frames[-1]] * (T - len(frames)))
    elif len(frames) > T:
        frames = frames[:T]
    return frames, orig_W, orig_H

## 2) BBox / ROI cropping utilities

3 modes:
1. **Annotation bbox** (if available)
2. **Fallback ROI crop** (center/upper-body-ish crop)
3. **Plug-in detector** (later): add MediaPipe / pose detection to produce bboxes


In [ ]:
# BBox handling
# The cleaned WLASL videos are already in a 256x256 coordinate space.
# For this baseline, we ALWAYS use the full-frame bbox so there is no
# possibility of scale / format mismatch.
FULL_BBOX_256 = FIXED_BBOX  # (0, 0, 256, 256)

def get_bbox_256(row=None):
    """Return the bbox to use for a sample (always full frame in this baseline)."""
    return FULL_BBOX_256

def center_roi_bbox(frame, *args, **kwargs):
    """Legacy fallback (kept for compatibility): full frame."""
    return FULL_BBOX_256

def clip_bbox(bbox, w, h):
    x1, y1, x2, y2 = bbox
    x1 = max(0, min(w-1, int(x1)))
    x2 = max(0, min(w,   int(x2)))
    y1 = max(0, min(h-1, int(y1)))
    y2 = max(0, min(h,   int(y2)))
    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2

def crop_with_bbox(frame, bbox):
    """Crop frame by bbox. With full-frame bbox, this is a no-op."""
    h, w = frame.shape[:2]
    bb = clip_bbox(bbox, w, h)
    if bb is None:
        return frame
    x1, y1, x2, y2 = bb
    return frame[y1:y2, x1:x2]

def parse_annotation_bbox(row):
    """
    Extract bbox from dataset row.
    Adjust this if your JSON field name is different.
    """
    if "bbox" in row and row["bbox"] is not None:
        return row["bbox"]
    return None


TARGET = 256

def xywh_to_xyxy(b):
    x, y, w, h = map(float, b)
    return [x, y, x + w, y + h]

def clamp_xyxy(b, W=TARGET, H=TARGET):
    x1, y1, x2, y2 = map(float, b)
    x1 = max(0, min(W, x1)); x2 = max(0, min(W, x2))
    y1 = max(0, min(H, y1)); y2 = max(0, min(H, y2))
    if x2 < x1: x1, x2 = x2, x1
    if y2 < y1: y1, y2 = y2, y1
    return [x1, y1, x2, y2]

def normalize_bbox_to_256(bbox, orig_W, orig_H):
    """
    Accepts bbox in one of:
      - normalized xyxy (0..1)
      - pixel xyxy in original resolution
      - pixel xywh in original resolution
    Returns xyxy bbox in 0..256 coordinates.
    """
    if bbox is None:
        return None

    vals = list(map(float, bbox))
    if len(vals) != 4:
        return None

    # Case 1: normalized (0..1)
    if max(vals) <= 1.5:
        x1, y1, x2, y2 = vals
        return clamp_xyxy([x1*TARGET, y1*TARGET, x2*TARGET, y2*TARGET])

    # Heuristic: treat as xywh if it fits
    x1, y1, x2, y2 = vals
    if (x1 + x2 <= orig_W + 5) and (y1 + y2 <= orig_H + 5):
        vals = xywh_to_xyxy(vals)

    # Scale from original size → 256
    sx, sy = TARGET / orig_W, TARGET / orig_H
    x1, y1, x2, y2 = vals
    return clamp_xyxy([x1*sx, y1*sy, x2*sx, y2*sy])

TARGET = 256

def clamp_xyxy(b, W=TARGET, H=TARGET):
    x1, y1, x2, y2 = map(float, b)
    x1 = max(0, min(W, x1)); x2 = max(0, min(W, x2))
    y1 = max(0, min(H, y1)); y2 = max(0, min(H, y2))
    if x2 < x1: x1, x2 = x2, x1
    if y2 < y1: y1, y2 = y2, y1
    return [x1, y1, x2, y2]

def center_crop_bbox(crop=192, W=TARGET, H=TARGET):
    crop = int(min(crop, W, H))
    x1 = (W - crop) // 2
    y1 = (H - crop) // 2
    return [x1, y1, x1 + crop, y1 + crop]

def motion_bbox_256(frames_bgr, thresh=18, min_area=600, pad=16):
    """
    Compute a bbox in 256-space from motion across frames.
    frames_bgr: list of BGR frames, each 256x256.

    Returns xyxy bbox (floats) or None.
    """
    if frames_bgr is None or len(frames_bgr) < 2:
        return None

    # grayscale stack
    grays = [cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY) for fr in frames_bgr]
    # use median frame as background estimate
    bg = np.median(np.stack(grays, axis=0).astype(np.float32), axis=0).astype(np.uint8)

    # accumulate motion mask
    acc = np.zeros_like(bg, dtype=np.uint8)
    for g in grays:
        diff = cv2.absdiff(g, bg)
        _, mask = cv2.threshold(diff, thresh, 255, cv2.THRESH_BINARY)
        acc = cv2.bitwise_or(acc, mask)

    # clean mask
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    acc = cv2.morphologyEx(acc, cv2.MORPH_OPEN, k, iterations=1)
    acc = cv2.morphologyEx(acc, cv2.MORPH_CLOSE, k, iterations=2)

    # find largest connected component / contour
    cnts, _ = cv2.findContours(acc, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None

    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    if area < min_area:
        return None

    x, y, w, h = cv2.boundingRect(c)
    x1 = x - pad
    y1 = y - pad
    x2 = x + w + pad
    y2 = y + h + pad

    return clamp_xyxy([x1, y1, x2, y2], W=TARGET, H=TARGET)

## 3) Demo: sample clips and visualize frames (before vs after crop)

This verifies that your pipeline can load videos, sample frames, and crop.

In [ ]:
if "bbox" not in clean_df.columns:
    print("No bbox column in this WLASL variant.")
else:
    print("bbox present %:", (clean_df["bbox"].notna().mean() * 100).round(2))
    # show a few bbox examples
    ex = clean_df.loc[clean_df["bbox"].notna(), ["gloss", "video_id", "bbox"]].head(10)
    ex

clean_df.loc[clean_df["bbox"].notna(), ["bbox"]].head(10)

In [ ]:
def draw_bbox(frame, bbox, thickness=2):
    fr = frame.copy()
    h, w = fr.shape[:2]
    bb = clip_bbox(bbox, w, h)
    if bb is None:
        return fr
    x1, y1, x2, y2 = bb
    # draw rectangle (red)
    fr = cv2.rectangle(fr, (x1, y1), (x2, y2), (255, 0, 0), thickness)
    return fr

In [ ]:
def show_frame_grid(frames, title, ncols=8):
    if len(frames) == 0:
        print(f"No frames to show for: {title}")
        return
    n = len(frames)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    plt.figure(figsize=(2.2*ncols, 2.2*nrows))
    for i, fr in enumerate(frames):
        ax = plt.subplot(nrows, ncols, i+1)
        ax.imshow(fr)
        ax.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

def demo_one_instance(row_idx, T=NUM_FRAMES_CONSTANT, use_loc=False, display_max_side=256,
                      use_annotation_bbox=True, fallback_center_crop=192):
    """Visualize a sample with the bbox/crop pipeline.

    - Frames are resized to 256x256 in read_frames_uniform.
    - If use_annotation_bbox=True, uses the dataset bbox (normalized into 256-space).
    - Otherwise, uses a center-crop bbox (default 192x192) to "zoom" consistently.
    - If bbox is missing/invalid, falls back to center-crop, then full-frame.
    """
    row = clean_df.loc[row_idx] if use_loc else clean_df.iloc[row_idx]
    vid = str(row.get("video_id", ""))
    gloss = row.get("gloss", "")
    vp = resolve_video_path(vid)
    if vp is None:
        print("Video not found:", vid)
        return

    # read frames (+ optionally original size)
    out = read_frames_uniform(vp, T=T)
    if isinstance(out, tuple) and len(out) == 3:
        frames, orig_W, orig_H = out
    else:
        frames = out
        orig_W, orig_H = None, None

    if len(frames) == 0:
        print("Could not read frames.")
        return

    # Helper: resize only for display (usually a no-op since frames are 256x256)
    def resize_for_display(frame, max_side=display_max_side):
        if max_side is None or cv2 is None:
            return frame
        h, w = frame.shape[:2]
        scale = max(h, w) / max_side
        if scale <= 1:
            return frame
        new_w = int(w / scale)
        new_h = int(h / scale)
        return cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Always trust the actual frame size we have (should be 256x256)
    H256, W256 = frames[0].shape[:2]

    def center_crop_bbox(crop=192, W=256, H=256):
        crop = int(min(crop, W, H))
        x1 = (W - crop) // 2
        y1 = (H - crop) // 2
        return [x1, y1, x1 + crop, y1 + crop]

    # --- choose bbox ---
    # Prefer motion bbox on the CLEANED frames (reliable in 256-space)
    bbox = motion_bbox_256(frames, thresh=18, min_area=600, pad=16)
    
    # Fallback: if motion bbox fails, use a consistent center crop
    if bbox is None:
        bbox = center_crop_bbox(crop=192)


    if use_annotation_bbox:
        raw_bbox = parse_annotation_bbox(row) if "parse_annotation_bbox" in globals() else None

        # If we have original dims, normalize from that into 256-space
        if raw_bbox is not None and orig_W is not None and orig_H is not None and "normalize_bbox_to_256" in globals():
            bbox = normalize_bbox_to_256(raw_bbox, orig_W, orig_H)
        else:
            # No orig dims available: assume bbox already matches current frame coords
            bbox = raw_bbox

    # If bbox missing/invalid, fall back to center crop (zoom)
    def _bbox_valid(b):
        if b is None: return False
        try:
            x1, y1, x2, y2 = map(float, b)
        except Exception:
            return False
        return (x2 > x1) and (y2 > y1)

    if not _bbox_valid(bbox):
        bbox = center_crop_bbox(crop=fallback_center_crop, W=W256, H=H256)

    # Final safety clamp to frame bounds (256-space)
    if "clamp_xyxy" in globals():
        bbox = clamp_xyxy(bbox, W=W256, H=H256)
    else:
        # minimal clamp if clamp_xyxy isn't defined
        x1, y1, x2, y2 = map(float, bbox)
        x1 = max(0, min(W256, x1)); x2 = max(0, min(W256, x2))
        y1 = max(0, min(H256, y1)); y2 = max(0, min(H256, y2))
        if x2 < x1: x1, x2 = x2, x1
        if y2 < y1: y1, y2 = y2, y1
        bbox = [x1, y1, x2, y2]

    # --- apply pipeline ---
    boxed = [draw_bbox(fr, bbox) for fr in frames]
    cropped = [crop_with_bbox(fr, bbox) for fr in frames]

    def bgr_to_rgb(fr):
        return cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)

    #frames_disp  = [bgr_to_rgb(resize_for_display(fr)) for fr in frames]
    boxed_disp   = [bgr_to_rgb(resize_for_display(fr)) for fr in boxed]
    cropped_disp = [bgr_to_rgb(resize_for_display(fr)) for fr in cropped]


    # Titles include bbox debug info
    #title_raw = f"RAW (256x256) — {gloss} ({vid})"
    title_box = f"RAW + BBOX — {gloss} | bbox={tuple(int(v) for v in bbox)}"
    title_crop = f"CROPPED — {gloss} | bbox={tuple(int(v) for v in bbox)}"

    #show_frame_grid(frames_disp,  title_raw)
    show_frame_grid(boxed_disp,   title_box)
    show_frame_grid(cropped_disp, title_crop)

In [ ]:
# Pick a few random *positional* indices that exist locally and demo them
if VIDEO_DIR is None or "video_id" not in clean_df.columns:
    print("Set VIDEO_DIR and ensure clean_df has video_id.")
elif cv2 is None:
    print("Install cv2/opencv-python to run the demo.")
else:
    candidate_pos = []
    sample_pool = clean_df.sample(min(2000, len(clean_df)), random_state=0)  # no reset_index
    for pos, row in enumerate(sample_pool.itertuples(index=False), start=0):
        vid = str(getattr(row, "video_id"))
        if resolve_video_path(vid) is not None:
            # convert sample_pool position back to clean_df position
            # easiest: just grab the actual positional index from sample_pool.index
            candidate_pos.append(int(sample_pool.index[pos]))
        #Just takes the first 7 to keep things simple for now
        if len(candidate_pos) >= 7:
            break

    print("Demo indices (label index):", candidate_pos)
    #This is where the ids of the videos shown below are patched through to our function
    #Change the params of the for loop if you want to see different words
    for idx in candidate_pos[0:4]:
        # use .loc instead of .iloc
        demo_one_instance(idx, T=NUM_FRAMES_CONSTANT, use_annotation_bbox=False, use_loc=True)

In [ ]:
# --------------------
# Config
# --------------------
SEED = 99
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
#torch.backends.cudnn.benchmark = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

T_FRAMES = 12           # N frames per sample (uniformly sampled)
MODEL_IMG_SIZE = 224     # after bbox crop, resize to this for CNN
BATCH_SIZE = 4
NUM_WORKERS = 4
LR = 1e-4
K = 100              # Top-K glosses to keep for experiments
EPOCHS = 15

# bbox fallbacks
CENTER_CROP_SIZE = 192   # in 256-space
MOTION_THRESH = 18
MOTION_MIN_AREA = 600
MOTION_PAD = 16

In [ ]:
def choose_bbox_256_for_clip(frames_256_bgr, row, orig_W=None, orig_H=None,
                             prefer_motion=True,
                             prefer_annot=False,
                             center_crop_size=CENTER_CROP_SIZE):
    """
    Returns a bbox in 256-space (xyxy floats) using:
      1) motion bbox (over frames)
      2) annotated bbox (normalized into 256-space)
      3) center crop bbox
      4) full frame
    """
    # 1) motion bbox 
    if prefer_motion:
        bb = motion_bbox_256(
            frames_256_bgr,
            thresh=MOTION_THRESH,
            min_area=MOTION_MIN_AREA,
            pad=MOTION_PAD
        )
        if bb is not None:
            return bb

    # 2) annotated bbox
    if prefer_annot:
        ann = parse_annotation_bbox(row)
        if ann is not None:
            # normalize into 256-space if needed
            try:
                if orig_W is not None and orig_H is not None:
                    bb = normalize_bbox_to_256(ann, orig_W, orig_H)
                else:
                    # if ann already normalized or already 256-space, normalize_bbox_to_256
                    # may still work; if it doesn't, we clamp below.
                    bb = normalize_bbox_to_256(ann, 256, 256)
                if bb is not None:
                    return bb
            except Exception:
                pass

    # 3) center crop
    try:
        bb = center_crop_bbox(crop=center_crop_size, W=256, H=256)
        if bb is not None:
            return bb
    except Exception:
        pass

    # 4) full frame
    return list(FULL_BBOX_256)

In [ ]:
def crop_resize_clip(frames_256_bgr, bbox_256_xyxy, out_size=MODEL_IMG_SIZE):
    """
    frames_256_bgr: list of BGR frames (256x256x3)
    bbox_256_xyxy: xyxy bbox in 256-space (floats ok)
    Returns: numpy array (T, out_size, out_size, 3) in RGB
    """
    cropped = []
    for fr in frames_256_bgr:
        fr_crop = crop_with_bbox(fr, bbox_256_xyxy)  # returns smaller HxWx3
        # resize to CNN input
        fr_crop = cv2.resize(fr_crop, (out_size, out_size), interpolation=cv2.INTER_AREA)
        # BGR -> RGB
        fr_crop = cv2.cvtColor(fr_crop, cv2.COLOR_BGR2RGB)
        cropped.append(fr_crop)
    return np.stack(cropped, axis=0)  # (T, H, W, 3)

In [ ]:
# -----------------------------------------------------------------------------
# Split helpers
# -----------------------------------------------------------------------------
def get_signer_column(df):
    cols = [c for c in df.columns if "signer" in c.lower()]
    if not cols:
        raise KeyError("Could not find a signer column (no column name contains 'signer').")
    return cols[0]

def make_split(df, mode="signer", train_frac=0.85, seed=SEED):
    """Return (train_df, val_df). mode in {'random','signer'}"""
    rng = np.random.RandomState(seed)

    if mode == "random":
        perm = rng.permutation(len(df))
        split = int(train_frac * len(df))
        tr = df.iloc[perm[:split]].reset_index(drop=True)
        va = df.iloc[perm[split:]].reset_index(drop=True)
        return tr, va

    if mode == "signer":
        signer_col = get_signer_column(df)
        signers = df[signer_col].astype(str).fillna("NA_SIGNER").unique().tolist()
        rng.shuffle(signers)

        split = int(train_frac * len(signers))
        tr_signers = set(signers[:split])
        va_signers = set(signers[split:])

        tr = df[df[signer_col].astype(str).fillna("NA_SIGNER").isin(tr_signers)].reset_index(drop=True)
        va = df[df[signer_col].astype(str).fillna("NA_SIGNER").isin(va_signers)].reset_index(drop=True)

        # sanity: no overlap
        assert set(tr[signer_col].astype(str)).isdisjoint(set(va[signer_col].astype(str)))
        return tr, va

    raise ValueError("mode must be 'random' or 'signer'")

In [ ]:
# -----------------------------------------------------------------------------
# BBox configuration
# -----------------------------------------------------------------------------
# By default, this notebook used a bbox chooser that prioritizes motion bboxes.
# For controlled experiments, we expose a simple switch:
#   - USE_BBOX=False  -> always full-frame (no bbox crop)
#   - USE_BBOX=True + BBOX_SOURCE="annot" -> use dataset annotations when available
# You can extend BBOX_SOURCE to "motion" later if you want.
USE_BBOX = False
BBOX_SOURCE = "annot"   # "annot" or "motion"

BBOX_CACHE = {}

def bbox_config_key():
    return (bool(USE_BBOX), str(BBOX_SOURCE))

def cached_bbox(frames_256_bgr, row, orig_W, orig_H):
    """Return a bbox in 256-space, cached per (video_id, bbox_config)."""
    vid = str(row["video_id"])
    key = (vid,) + bbox_config_key()
    if key in BBOX_CACHE:
        return BBOX_CACHE[key]

    if not USE_BBOX:
        bb = list(FULL_BBOX_256)  # full frame, no crop
    else:
        if BBOX_SOURCE == "annot":
            bb = choose_bbox_256_for_clip(
                frames_256_bgr, row, orig_W=orig_W, orig_H=orig_H,
                prefer_motion=False,
                prefer_annot=True,
                center_crop_size=CENTER_CROP_SIZE
            )
        elif BBOX_SOURCE == "motion":
            bb = choose_bbox_256_for_clip(
                frames_256_bgr, row, orig_W=orig_W, orig_H=orig_H,
                prefer_motion=True,
                prefer_annot=False,
                center_crop_size=CENTER_CROP_SIZE
            )
        else:
            raise ValueError(f"Unknown BBOX_SOURCE={BBOX_SOURCE!r}")

    BBOX_CACHE[key] = bb
    return bb

In [ ]:
# Normalization for ResNet backbones
resnet_norm = tvt.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

class ASLClipDataset(Dataset):
    def __init__(self, df, T=T_FRAMES, out_size=MODEL_IMG_SIZE, training=True):
        self.df = df.reset_index(drop=True)
        self.T = T
        self.out_size = out_size
        self.training = training

        self.aug = tvt.Compose([
            tvt.RandomResizedCrop(self.out_size, scale=(0.75, 1.0)),
            tvt.RandomHorizontalFlip(p=0.5),
            tvt.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
            tvt.RandomGrayscale(p=0.1),
            tvt.RandomAffine(degrees=10, translate=(0.05, 0.05)),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        vid = str(row["video_id"])
        gloss = str(row["gloss"])
        y = gloss_to_idx[gloss]

        vp = resolve_video_path(vid)
        if vp is None:
            raise FileNotFoundError(f"Video not found for id={vid}")

        # Read uniformly T frames at 256px
        frames_256_bgr, orig_W, orig_H = read_frames_uniform(vp, T=self.T, target_size=256)
        if len(frames_256_bgr) == 0:
            raise RuntimeError(f"Could not read frames for id={vid}")

        # Crop and resize using cached bbox
        bbox = cached_bbox(frames_256_bgr, row, orig_W, orig_H)
        clip_rgb = crop_resize_clip(frames_256_bgr, bbox, out_size=self.out_size)  # (T,H,W,3)

        # Convert to torch tensor (T, C, H, W)
        clip = torch.from_numpy(clip_rgb).float() / 255.0
        clip = clip.permute(0, 3, 1, 2)  # T,C,H,W

        # Apply augmentation + normalization framewise
        frames_out = []
        for t in range(clip.shape[0]):
            fr = clip[t]
            if self.training:
                fr = self.aug(fr)  # augment
            fr = resnet_norm(fr)  # normalize
            frames_out.append(fr)

        # Stack back to tensor (T,C,H,W)
        clip = torch.stack(frames_out, dim=0)

        return clip, torch.tensor(y, dtype=torch.long)

In [ ]:
# -----------------------------------------------------------------------------
# Build datasets/loaders for an experiment
# -----------------------------------------------------------------------------
def build_loaders(df, split_mode="signer", K=100, seed=SEED):
    # top-K gloss restriction
    counts = df["gloss"].value_counts()
    top_glosses = counts.head(K).index
    df_k = df[df["gloss"].isin(top_glosses)].copy()

    # rebuild label mapping for this experiment
    global glosses, gloss_to_idx, idx_to_gloss, NUM_CLASSES
    glosses = sorted(df_k["gloss"].astype(str).unique().tolist())
    gloss_to_idx = {g:i for i,g in enumerate(glosses)}
    idx_to_gloss = {i:g for g,i in gloss_to_idx.items()}
    NUM_CLASSES = len(glosses)
    print(f"K={K} | rows={len(df_k)} | NUM_CLASSES={NUM_CLASSES} | split_mode={split_mode}")

    # split
    train_df, val_df = make_split(df_k, mode=split_mode, train_frac=0.85, seed=seed)

    # balanced sampler (optional but usually stabilizes top-K training)
    counts_tr = train_df["gloss"].value_counts()
    weights = train_df["gloss"].map(lambda g: 1.0 / counts_tr[g]).values
    sampler = WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)

    train_ds = ASLClipDataset(train_df, training=True)
    val_ds   = ASLClipDataset(val_df, training=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
        prefetch_factor=4,
        drop_last=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
        prefetch_factor=4
    )
    return train_loader, val_loader

In [ ]:
class CNNFrameAvg(nn.Module):
    """Per-frame CNN backbone + temporal average pooling."""
    def __init__(self, num_classes, backbone="resnet50", dropout=0.2):
        super().__init__()

        if backbone == "resnet50":
            net = torchvision.models.resnet50(
                weights=torchvision.models.ResNet50_Weights.DEFAULT
            )
        elif backbone == "resnet18":
            net = torchvision.models.resnet18(
                weights=torchvision.models.ResNet18_Weights.DEFAULT
            )
        else:
            raise ValueError("backbone must be 'resnet50' or 'resnet18'")

        feat_dim = net.fc.in_features
        net.fc = nn.Identity()
        self.cnn = net

        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(feat_dim, num_classes)
        )

    def forward(self, clips):
        # clips: (B, T, C, H, W)
        B, T, C, H, W = clips.shape
        x = clips.view(B*T, C, H, W)
        feats = self.cnn(x)              # (B*T, D)
        feats = feats.view(B, T, -1)     # (B, T, D)
        feats = feats.mean(dim=1)        # (B, D)
        return self.head(feats)          # (B, num_classes)

class CNNLSTM(nn.Module):
    def __init__(self, num_classes, backbone="resnet50", hidden=512, layers=2, bidir=True, dropout=0.3):
        super().__init__()

        if backbone == "resnet50":
            net = torchvision.models.resnet50(
                weights=torchvision.models.ResNet50_Weights.DEFAULT
            )
        else:
            raise ValueError("Only resnet50 implemented here")

        feat_dim = net.fc.in_features
        net.fc = nn.Identity()
        self.cnn = net

        self.lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            bidirectional=bidir,
            dropout=dropout if layers > 1 else 0.0
        )

        out_dim = hidden * (2 if bidir else 1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(out_dim, num_classes)
        )

    def forward(self, clips):
        # clips: (B,T,C,H,W)
        B, T, C, H, W = clips.shape
        x = clips.view(B*T, C, H, W)
        feats = self.cnn(x)              # (B*T, D)
        feats = feats.view(B, T, -1)     # (B, T, D)

        lstm_out, _ = self.lstm(feats)   # (B, T, out_dim)
        pooled = lstm_out.mean(dim=1)    # mean over time
        logits = self.classifier(pooled)
        return logits

In [ ]:
# -----------------------------------------------------------------------------
# Experiment runner
# -----------------------------------------------------------------------------
def build_model(model_type, num_classes):
    if model_type == "cnn":
        return CNNFrameAvg(num_classes=num_classes, backbone="resnet50", dropout=0.2)
    if model_type == "cnnlstm":
        return CNNLSTM(num_classes=num_classes, backbone="resnet50", hidden=512, layers=2, bidir=True, dropout=0.3)
    raise ValueError("model_type must be 'cnn' or 'cnnlstm'")

def run_experiment(name, model_type, split_mode, use_bbox, bbox_source="motion", K=100, epochs=EPOCHS, seed=SEED):
    global USE_BBOX, BBOX_SOURCE, BBOX_CACHE
    USE_BBOX = bool(use_bbox)
    BBOX_SOURCE = str(bbox_source)
    BBOX_CACHE = {}  # clear cache between experiments so memory doesn't snowball

    print("\n" + "="*110)
    print(f"EXPERIMENT: {name}")
    print(f"model={model_type} | split={split_mode} | use_bbox={USE_BBOX} ({BBOX_SOURCE}) | K={K} | epochs={epochs}")

    # loaders
    train_loader, val_loader = build_loaders(clean_df, split_mode=split_mode, K=K, seed=seed)

    # model + optim
    model = build_model(model_type, NUM_CLASSES).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=2e-4)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)

    # train
    history = []
    for epoch in range(1, epochs+1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, opt, crit, scaler)
        va_loss, va_acc1, va_acc5 = evaluate(model, val_loader, crit)
        scheduler.step(va_loss)

        row = {
            "experiment": name,
            "epoch": epoch,
            "train_loss": tr_loss,
            "train_acc": tr_acc,
            "val_loss": va_loss,
            "val_top1": va_acc1,
            "val_top5": va_acc5,
            "lr": opt.param_groups[0]["lr"],
        }
        history.append(row)

        print(
            f"Epoch {epoch:02d} | "
            f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {va_loss:.4f} top1 {va_acc1:.4f} top5 {va_acc5:.4f} | "
            f"lr {opt.param_groups[0]['lr']:.2e}"
        )

    return pd.DataFrame(history)

In [ ]:
@torch.no_grad()
def evaluate(model, loader, crit, topk=(1,5)):
    model.eval()
    total_loss, total, correct1, correct5 = 0.0, 0, 0, 0

    for clips, y in loader:
        clips = clips.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        logits = model(clips)
        loss = crit(logits, y)

        total_loss += float(loss.item()) * y.size(0)
        total += y.size(0)

        # top-k
        maxk = max(topk)
        _, pred = logits.topk(maxk, dim=1, largest=True, sorted=True)  # (B, maxk)
        correct = pred.eq(y.view(-1,1))  # (B, maxk)

        correct1 += int(correct[:, :1].sum().item())
        correct5 += int(correct[:, :5].sum().item())

    return total_loss/total, correct1/total, correct5/total

In [ ]:
def train_one_epoch(model, loader, opt, crit, scaler, log_every=20):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    t_epoch = time.time()

    for step, (clips, y) in enumerate(loader, 1):
        if step == 1:
            print("First batch arrived:", clips.shape)

        clips = clips.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
            logits = model(clips)
            loss = crit(logits, y)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        total_loss += float(loss.item()) * y.size(0)
        total_correct += int((logits.argmax(dim=1) == y).sum().item())
        total += y.size(0)

        if step % log_every == 0:
            dt = time.time() - t_epoch
            print(f" step {step:4d}/{len(loader)} | loss {loss.item():.4f} | elapsed {dt:.1f}s")

    return total_loss/total, total_correct/total

In [ ]:
# Quick loader sanity check (optional)
_tmp_train_loader, _tmp_val_loader = build_loaders(clean_df, split_mode="random", K=K, seed=SEED)

t0 = time.time()
batch = next(iter(_tmp_train_loader))
t1 = time.time()

clips, y = batch
print("Loaded one batch.")
print("clips:", clips.shape, clips.dtype)
print("y:", y.shape, y.dtype)
print("seconds:", round(t1 - t0, 2))

print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# -----------------------------------------------------------------------------
# Validation suite (requested configs)
# -----------------------------------------------------------------------------
df_cnn_random = []
df_cnn_signer = []
df_cnn_signer_bbox = []
df_cnnlstm_signer_bbox = []

df_cnn_random = run_experiment(
    name="CNN | random split | no bbox",
    model_type="cnn",
    split_mode="random",
    use_bbox=False,
    K=K,
    epochs=EPOCHS,
    seed=SEED
)

df_cnn_signer = run_experiment(
    name="CNN | signer split | no bbox",
    model_type="cnn",
    split_mode="signer",
    use_bbox=False,
    K=K,
    epochs=EPOCHS,
    seed=SEED
)

df_cnn_signer_bbox = run_experiment(
    name="CNN | signer split | bbox(motion)",
    model_type="cnn",
    split_mode="signer",
    use_bbox=True,
    bbox_source="motion",
    K=K,
    epochs=EPOCHS,
    seed=SEED
)

df_cnnlstm_signer_bbox = run_experiment(
    name="CNNLSTM | signer split | bbox(motion)",
    model_type="cnnlstm",
    split_mode="signer",
    use_bbox=True,
    bbox_source="motion",
    K=K,
    epochs=EPOCHS,
    seed=SEED
)

df_cnn_random_df = pd.concat(df_cnn_random, ignore_index=True)
df_cnn_random_df.tail(10)

df_cnn_signer_df = pd.concat(df_cnn_signer, ignore_index=True)
df_cnn_signer_df.tail(10)

df_cnn_signer_bbox_df = pd.concat(df_cnn_signer_bbox, ignore_index=True)
df_cnn_signer_bbox_df.tail(10)

df_cnnlstm_signer_bbox_df = pd.concat(df_cnnlstm_signer_bbox, ignore_index=True)
df_cnnlstm_signer_bbox_df.tail(10)